In [ ]:
!pip install opencv-python-headless
!pip install numpy

In [ ]:
import cv2
import numpy as np
import time

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# install yolov8

!pip install ultralytics

In [ ]:
# import yolo

from ultralytics import YOLO

In [ ]:
# check yolo version
!yolo version

In [ ]:
# load the yolo pretrained model

# Load BIG model (YOLOv8x)
model = YOLO("yolov8x.pt")
print("Loaded YOLOv8x model.")

In [ ]:
# TRAINING

model.train(
    data="/content/drive/My Drive/Colab Notebooks/tea_looper_detection/extracted_data_tea_looper/data.yaml",
    epochs=10,
    imgsz=510,
    batch=16,
    name="insect_yolov8x_soumili_v1",
    project="/content/drive/My Drive/Colab Notebooks/tea_looper_detection_classification_final/insect_output_yolov8x_v4"
)

print("Training completed!")

In [ ]:
#evaluate and run inference

#evaluate on validation data
insect_model = YOLO("/content/drive/My Drive/Colab Notebooks/tea_looper_detection_classification_final/insect_output_yolov8x_v4/insect_yolov8x_soumili_v1/weights/best.pt")
insect_model.val(
    data="/content/drive/My Drive/Colab Notebooks/tea_looper_detection/extracted_data_tea_looper/data.yaml",
    project="/content/drive/My Drive/Colab Notebooks/tea_looper_detection_classification_final/insect_detection_output_val_v3",  # your custom base folder
    name="validation_results_v2"  # subfolder name inside that
)

#run inference on test images
results = insect_model.predict(source="/content/drive/My Drive/Colab Notebooks/tea_looper_detection/extracted_data_tea_looper/valid/images", conf=0.25)

In [ ]:
# use the trained model for detection

insect_model = YOLO("/content/drive/My Drive/Colab Notebooks/tea_looper_detection_classification_final/insect_output_yolov8x_v4/insect_yolov8x_soumili_v1/weights/best.pt")
results = insect_model.predict(
    source="/content/drive/My Drive/Colab Notebooks/tea_looper_detection/test_img/tea_insect_detection",
    save=True,
    project="/content/drive/My Drive/Colab Notebooks/tea_looper_detection_classification_final/looper_detection_output_test_v3/results",  #custom folder
    name="test_run",  #subfolder name
    exist_ok=True  # avoid errors if folder already exists
)

print(" Results saved to:", results[0].save_dir)

In [ ]:
!pip install -U ultralytics

from ultralytics import YOLO
import os, shutil, random

In [ ]:
# DATASET SHUFFLING

DATASET_SRC = "/content/drive/MyDrive/Colab Notebooks/tea_looper_detection_classification_final/classification"  # CHANGE THIS
DATASET_DST = "/content/drive/MyDrive/Colab Notebooks/tea_looper_detection_classification_final/insect_classification_yolo_v1"  # OUTPUT FOLDER

# Create clean folders
for split in ["train", "val", "test"]:
    for folder in ["train", "val", "test"]:
        os.makedirs(f"{DATASET_DST}/{split}", exist_ok=True)

# Clear DST completely (safe rebuild)
if os.path.exists(DATASET_DST):
    shutil.rmtree(DATASET_DST)
os.makedirs(DATASET_DST)

SPLITS = {"train": 0.7, "val": 0.2, "test": 0.1}

print("Shuffling dataset...")

In [ ]:
# Loop through classes
for cls in os.listdir(DATASET_SRC):
    cls_path = os.path.join(DATASET_SRC, cls)
    if not os.path.isdir(cls_path):
        continue

    images = [os.path.join(cls_path, img) for img in os.listdir(cls_path)]
    random.shuffle(images)

    # compute split sizes
    n = len(images)
    n_train = int(n * SPLITS["train"])
    n_val = int(n * SPLITS["val"])
    n_test = n - n_train - n_val
   # assign files
    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train+n_val],
        "test": images[n_train+n_val:]
    }

    # Copy images
    for split_name, img_list in splits.items():
        split_folder = f"{DATASET_DST}/{split_name}/{cls}"
        os.makedirs(split_folder, exist_ok=True)

        for img_file in img_list:
            shutil.copy(img_file, split_folder)

print("Dataset shuffled and split successfully!")
print("New dataset:", DATASET_DST)


In [ ]:
# TRAIN YOLOv8x CLASSIFIER

model = YOLO("yolov8x-cls.pt")  # big classification model

results = model.train(
    data=DATASET_DST,
    epochs=10,
    imgsz=224,     # standard size for classification
    batch=32,
    name="insect_cls_yolov8x_v1",
    project="/content/drive/MyDrive/Colab Notebooks/tea_looper_detection_classification_final/insect_cls_output_v2"
)

print("Training finished!")


In [ ]:
# VALIDATION

model = YOLO(
    "/content/drive/MyDrive/Colab Notebooks/tea_looper_detection_classification_final/insect_cls_output_v2/insect_cls_yolov8x_v1/weights/best.pt"
)

model.val(
    data=DATASET_DST,
    project="/content/drive/MyDrive/Colab Notebooks/tea_looper_detection_classification_final/insect_cls_output_v1/val_results",
    name="cls_val_v1"
)

print("Validation done.")


In [ ]:
# INFERENCE ON NEW IMAGES

results = model.predict(
    source="/content/drive/MyDrive/Colab Notebooks/tea_looper_classification/test_classification",  # folder or single image
    save=True,
    project="/content/drive/MyDrive/Colab Notebooks/tea_looper_detection_classification_final/insect_cls_output_v1/test_results",
    name="test_run_v1",
    exist_ok=True
)

print("Results saved to:", results[0].save_dir)

In [ ]:
from ultralytics import YOLO
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import os

In [ ]:
# load model and class names

model_path = "/content/drive/MyDrive/Colab Notebooks/tea_looper_detection_classification_final/insect_cls_output_v2/insect_cls_yolov8x_v1/weights/best.pt"

model = YOLO(model_path)
classes = model.names


In [ ]:
# transform

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def predict_image(img_path):
    print(f"\n Predicting for image: {img_path}")

    # Read image
    image = Image.open(img_path).convert("RGB")

    # Preprocess
    img_tensor = val_transform(image).unsqueeze(0).to(device)

    # Model inference
    model.eval()
    with torch.no_grad():
        result = model(img_tensor, verbose=False)[0]
        probs = result.probs  # already softmaxed

    # Get prediction
    top_class = probs.top1
    top_conf  = probs.top1conf.item() * 100

    predicted_label = classes[top_class]

    print(f" Predicted Insect: {predicted_label}")
    print(f" Confidence: {top_conf:.2f}%")

    # Display image
    plt.figure(figsize=(4,4))
    plt.imshow(image)
    plt.title(f"{predicted_label} ({top_conf:.1f}%)")
    plt.axis("off")
    plt.show()

In [ ]:
# run inference

predict_image("/content/drive/MyDrive/Colab Notebooks/tea_looper_classification/test_classification/img (16).JPG")